# Disneyland RAG System Demo

This notebook demonstrates the RAG system for answering questions about Disneyland visitor reviews.

## Setup

Load environment, instantiate embeddings and LLM.

In [ ]:
import sys
from pathlib import Path

# Add src to path
sys.path.insert(0, str(Path.cwd().parent / "src"))

from rag.config import DATA_PATH, EMBEDDING_MODEL_NAME, LLM_MODEL_NAME, LLM_TEMPERATURE, LLM_MAX_TOKENS, LLM_TIMEOUT
from rag.embeddings import SentenceTransformerEmbeddings
from rag.ingest import load_reviews
from rag.vectorstore import get_or_build_collection
from rag.chain import ask
from langchain_litellm import ChatLiteLLM

print(f"Data path: {DATA_PATH}")
print(f"Embedding model: {EMBEDDING_MODEL_NAME}")
print(f"LLM model: {LLM_MODEL_NAME}")

## Load and Embed Data

This cell loads reviews from CSV and embeds them into ChromaDB.
On first run, this takes 3-5 minutes. Subsequent runs load from disk instantly.

In [2]:
# Load reviews
print("Loading reviews from CSV...")
documents = load_reviews(DATA_PATH)
print(f"Loaded {len(documents)} reviews")

# Initialize embeddings
print(f"\nInitializing embeddings with {EMBEDDING_MODEL_NAME}...")
embeddings = SentenceTransformerEmbeddings()

# Build or load ChromaDB collection
print("Building/loading ChromaDB collection...")
collection = get_or_build_collection(documents, embeddings)
print(f"Collection size: {collection.count()} documents")

Loading reviews from CSV...
  (Skipped 20 duplicate review IDs)
Loaded 42636 reviews

Initializing embeddings with all-MiniLM-L6-v2...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 16014.73it/s]


Building/loading ChromaDB collection...

📚 EMBEDDINGS LOADED FROM CACHE (instant)
   Location: /home/syaramionak/Projects/rag-system-poc/chroma_db
   Documents: 42,636
   Status: Ready to use

Collection size: 42636 documents


## Initialize LLM

Create a ChatLiteLLM instance pointing to the LiteLLM proxy.

In [ ]:
import os

proxy_url = os.getenv("LITELLM_PROXY_URL", "https://litellm.gke-prod.linnovate.net")
api_key = os.getenv("LITELLM_MASTER_KEY")

print(f"Proxy URL: {proxy_url}")
print(f"API Key: {'***' if api_key else 'NOT SET'}")

llm = ChatLiteLLM(
    model=LLM_MODEL_NAME,
    api_base=proxy_url,
    api_key=api_key,
    temperature=LLM_TEMPERATURE,
    max_tokens=LLM_MAX_TOKENS,
    timeout=LLM_TIMEOUT,
)
print("\nLLM initialized successfully")
print(f"Temperature: {LLM_TEMPERATURE}, Max tokens: {LLM_MAX_TOKENS}")

## Example 1: Australia visitors at HongKong

Question: "What do visitors from Australia say about Disneyland in HongKong?"

In [4]:
question_1 = "What do visitors from Australia say about Disneyland in HongKong?"
print(f"Question: {question_1}")
print("\n" + "="*60)

answer_1 = ask(question_1, collection, embeddings, llm, auto_extract_filters=True, n_results=10)
print("Answer:")
print(answer_1)

Question: What do visitors from Australia say about Disneyland in HongKong?

Answer:
Visitors from Australia generally have a positive opinion about Hong Kong Disneyland. Many highlight the park as a joyful and magical place, with friendly staff, clean facilities, and enjoyable rides and shows. Several reviewers appreciate that it is close to Australia, making it a convenient option for a Disney experience. 

Some Australian visitors note that Hong Kong Disneyland is smaller than the US Disneyland parks but feel it is well-suited for families with young children. A few mention that the park has grown over the years with new lands added, enhancing the experience.

However, there are some mixed opinions as well. Long queues and overcrowding can be an issue, and a few mention that audio on some rides is in Cantonese, which could be a downside for non-Chinese speakers. The food is often described as overpriced and average in quality.

Overall, many Australian visitors highly recommend visi

## Example 2: Spring season visits

Question: "Is spring a good time to visit Disneyland?"

In [5]:
question_2 = "Is spring a good time to visit Disneyland?"
print(f"Question: {question_2}")
print("\n" + "="*60)

answer_2 = ask(question_2, collection, embeddings, llm, auto_extract_filters=True, n_results=10)
print("Answer:")
print(answer_2)

Question: Is spring a good time to visit Disneyland?

Answer:
Based on the visitor reviews, spring can be both a good and a busy time to visit Disneyland. Several reviewers mention that spring, particularly March and May, often has great weather — not too hot or cold — making it pleasant for a visit. However, spring break tends to be very crowded, with packed lines and busy paths.

Some tips from visitors to have a better experience during spring include:
- Visiting on weekdays rather than weekends.
- Using Fast Pass options and apps to check wait times.
- Avoiding peak spring break weeks if possible.

So, spring is a good time for the weather and overall experience if you plan carefully, but be prepared for crowds, especially during spring break.


## Example 3: California in June

Question: "Is Disneyland California usually crowded in June?"

In [6]:
question_3 = "Is Disneyland California usually crowded in June?"
print(f"Question: {question_3}")
print("\n" + "="*60)

answer_3 = ask(question_3, collection, embeddings, llm, auto_extract_filters=True, n_results=30)
print("Answer:")
print(answer_3)

Question: Is Disneyland California usually crowded in June?

Answer:
Based on the visitor reviews provided, Disneyland California is usually quite crowded in June. Multiple reviews mention the large crowds, with one reviewer describing them as "huge and at times overbearing" and others noting that the park feels "too small for the number of folks" visiting. Visitors often recommend going early when the park opens to take advantage of shorter lines. Despite the crowds, many visitors still enjoy their experience at Disneyland in June.


## Example 4: Staff friendliness in Paris

Question: "Is the staff in Paris friendly?"

In [7]:
question_4 = "Is the staff in Paris friendly?"
print(f"Question: {question_4}")
print("\n" + "="*60)

answer_4 = ask(question_4, collection, embeddings, llm, auto_extract_filters=True, n_results=30)
print("Answer:")
print(answer_4)

Question: Is the staff in Paris friendly?

Answer:
The reviews about the friendliness of the staff at Disneyland Paris are mixed:

- Several visitors from the United Kingdom and other countries mentioned that the staff were unfriendly, rude, and unhelpful. For example, one UK visitor in 2011 said the staff were rude, and another in 2015 said they did not meet a single friendly member of staff. Some also mentioned specific negative experiences, like a Starbucks employee being rude.
- On the other hand, some visitors praised the staff for being friendly, helpful, and going above and beyond. A UK visitor in 2013 said the staff were much friendlier and more helpful during their visit. Another UK visitor in 2013 mentioned very helpful staff who assisted with special needs. A visitor from the United States in 2015 commented that the staff were amazing and very helpful to children.
- A few reviews noted that the friendliness might depend on management or specific times, implying some variatio

## Debug: Inspect Retrieved Documents

For a given question, see what documents are retrieved before they go to the LLM.

In [ ]:
from rag.filter_parser import extract_filters
from rag.retriever import retrieve

debug_question = "Is Disneyland California usually crowded in June?"
print(f"Debug question: {debug_question}")

# Extract filters
filters = extract_filters(debug_question, llm)
print(f"\nExtracted filters: {filters}")

# Retrieve documents with all filters
retrieved_docs = retrieve(
    debug_question,
    collection,
    embeddings,
    n_results=30,
    branch=filters.get("branch"),
    reviewer_location=filters.get("reviewer_location"),
    season=filters.get("season"),
    min_rating=filters.get("min_rating"),
    year_month=filters.get("year_month"),
    prefer_recent=filters.get("prefer_recent"),
)
print(f"\nRetrieved {len(retrieved_docs)} documents:")
print("\n" + "-"*60 + "\n")
for i, doc in enumerate(retrieved_docs, 1):
    print(f"Document {i}:")
    print(f"  Metadata: {doc.metadata}")
    print(f"  Text: {doc.page_content[:200]}...")
    print()